In [1]:
from pathlib import Path
import pandas as pd

region_groups_data = pd.read_csv("../data/raw/ibe010-migration-by-country-groups-2016-2025-raw.csv", skiprows=2)

region_groups_data.head()

,Rādītāji,Valstu grupa,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Iebrauca Latvijā,Pavisam,8345,9916,10909,11223,8840,12689,38708,15629,14701,13826
1,Iebrauca Latvijā,ES-27 no 2020. g.,2254,2356,2294,2531,2506,3696,4080,4316,3992,3349
2,Iebrauca Latvijā,ES-28,4780,4841,4488,4727,4469,6202,6852,7122,6830,2146
3,Iebrauca Latvijā,ES kandidātvalstis,32,77,106,88,74,176,24857,2739,3061,3335
4,Iebrauca Latvijā,EBTA,185,179,246,254,222,364,423,502,499,364


In [2]:
region_groups_data.shape

(14, 12)

In [3]:
region_groups_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rādītāji      14 non-null     str  
 1   Valstu grupa  14 non-null     str  
 2   2016          14 non-null     int64
 3   2017          14 non-null     int64
 4   2018          14 non-null     int64
 5   2019          14 non-null     int64
 6   2020          14 non-null     int64
 7   2021          14 non-null     int64
 8   2022          14 non-null     int64
 9   2023          14 non-null     int64
 10  2024          14 non-null     int64
 11  2025          14 non-null     int64
dtypes: int64(10), str(2)
memory usage: 1.4 KB


In [4]:
identifier_columns = ["Rādītāji", "Valstu grupa"]

for column in identifier_columns:
    print(region_groups_data[column].value_counts(dropna=False))

Rādītāji
Iebrauca Latvijā        7
Izbrauca no Latvijas    7
Name: count, dtype: int64
Valstu grupa
Pavisam               2
ES-27 no 2020. g.     2
ES-28                 2
ES kandidātvalstis    2
EBTA                  2
NVS                   2
Pārējās valstis       2
Name: count, dtype: int64


In [5]:
print("Count of missing values:")
print(region_groups_data.isna().sum())

print("Count of empty strings:")
print(region_groups_data[identifier_columns].eq("").sum())

Count of missing values:
Rādītāji        0
Valstu grupa    0
2016            0
2017            0
2018            0
2019            0
2020            0
2021            0
2022            0
2023            0
2024            0
2025            0
dtype: int64
Count of empty strings:
Rādītāji        0
Valstu grupa    0
dtype: int64


In [6]:
for column in identifier_columns:
    stripped_values = region_groups_data[column].astype("string").str.strip()

    whitespace_count = stripped_values.ne(region_groups_data[column].astype("string")).sum()

    print(f"{column}: {whitespace_count} values with leading or trailing whitespace.")

Rādītāji: 0 values with leading or trailing whitespace.
Valstu grupa: 0 values with leading or trailing whitespace.


In [7]:
region_groups_data[identifier_columns].duplicated().sum()

np.int64(0)

In [8]:
year_columns = region_groups_data.columns[2:]

region_groups_melted = region_groups_data.melt(
    id_vars=identifier_columns,
    value_vars=year_columns,
    var_name="Gads",
    value_name="Skaits"
)

region_groups_melted.head()

,Rādītāji,Valstu grupa,Gads,Skaits
0,Iebrauca Latvijā,Pavisam,2016,8345
1,Iebrauca Latvijā,ES-27 no 2020. g.,2016,2254
2,Iebrauca Latvijā,ES-28,2016,4780
3,Iebrauca Latvijā,ES kandidātvalstis,2016,32
4,Iebrauca Latvijā,EBTA,2016,185


In [9]:
region_groups_melted["Gads"] = region_groups_melted["Gads"].astype("int64")

In [10]:
region_groups_melted.shape

(140, 4)

In [11]:
region_groups_melted.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rādītāji      140 non-null    str  
 1   Valstu grupa  140 non-null    str  
 2   Gads          140 non-null    int64
 3   Skaits        140 non-null    int64
dtypes: int64(2), str(2)
memory usage: 4.5 KB


In [12]:
print("All counts are non-negative:")
print((region_groups_melted["Skaits"] >= 0).all())

print("All counts are whole numbers:")
print((region_groups_melted["Skaits"] % 1 == 0).all())

All counts are non-negative:
True
All counts are whole numbers:
True


In [13]:
checks_to_compare = [("Iebrauca Latvijā", "Pavisam", 2016, 8345),
                     ("Iebrauca Latvijā", "Pavisam", 2025, 13826),
                     ("Izbrauca no Latvijas", "Pavisam", 2016, 20574),
                     ("Izbrauca no Latvijas", "Pavisam", 2025, 15117)
]

for indicator, country_group, year, expected_count in checks_to_compare:
    actual_count = region_groups_melted.loc[
        (region_groups_melted["Rādītāji"] == indicator)
        & (region_groups_melted["Valstu grupa"] == country_group)
        & (region_groups_melted["Gads"] == year),
        "Skaits"
    ].item()

    print(f"{indicator} | {country_group} | {year}: expected {expected_count}, found {actual_count}")

Iebrauca Latvijā | Pavisam | 2016: expected 8345, found 8345
Iebrauca Latvijā | Pavisam | 2025: expected 13826, found 13826
Izbrauca no Latvijas | Pavisam | 2016: expected 20574, found 20574
Izbrauca no Latvijas | Pavisam | 2025: expected 15117, found 15117
